In [ ]:
import gymnasium as gym, torch, torch.nn as nn, numpy as np
import matplotlib.pyplot as plt

env = gym.make("CartPole-v1")
STATE_DIM = env.observation_space.shape[0]   # 4
N_ACTIONS  = env.action_space.n               # 2

# ─── TODO 1: Define tu WorldModel ────────────────────────────────────────────
class WorldModel(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        # TODO: crea self.transition  (state_dim + n_actions → state_dim)
        # TODO: crea self.reward_head (state_dim + n_actions → 1)
        # Hint: usa nn.Sequential con 2 capas ocultas de 256 y ReLU
        pass

    def forward(self, s, a_onehot):
        # TODO: concatena x = [s, a_onehot] y predice residual:
        # Retorna: (s + self.transition(x), self.reward_head(x))
        pass

# ─── TODO 2: Función de recolección de datos ─────────────────────────────────
def collect_data(n_episodes=300):
    # TODO: recolecta (s, a, r, s') y retorna tensores (S, A, R, S2)
    # Importante: si terminated es True (poste cae), asigna r = -10.0
    # para que el WorldModel aprenda a penalizar estados de caída.
    pass

# ─── TODO 3: Loop de entrenamiento ───────────────────────────────────────────
def train_world_model(wm, S, A, R, S2, epochs=60):
    # TODO: implementa el loop de entrenamiento con Adam + MSELoss
    # Imprime loss cada 10 epochs
    pass


In [ ]:
# ─── TODO 4: Validación con hold-out ────────────────────────────────────────
def evaluate_model(wm, S_test, A_test, R_test, S2_test):
    """
    Calcula y retorna:
    - loss_transition: MSE entre s' predicho vs real
    - loss_reward:     MSE entre r predicho vs real
    - mae_per_dim:     Error absoluto medio por cada dimensión de s'
    """
    # TODO: implementa la evaluación
    # Hint: usa torch.no_grad() y calcula las métricas
    pass

# Una vez implementado, deberías obtener algo como:
# >> Loss transición test: 0.001
# >> Loss recompensa test: 0.025
# >> MAE por dimensión: [0.001, 0.008, 0.002, 0.012]
# (el componente 3 = velocidad angular suele ser el más difícil)


In [ ]:
# ─── TODO 5: Medir error de predicción vs horizonte H ────────────────────────
def measure_compounding_error(wm, env, H_max=20, n_trials=50):
    """
    Para cada H en 1..H_max:
      - Ejecuta n_trials trayectorias REALES de H pasos con política aleatoria
      - Simula las mismas trayectorias con el World Model desde s_0
      - Calcula el error promedio ||s_H_real - s_H_simulado||_2

    Retorna: errors_por_H (lista de longitud H_max)
    """
    errors = []
    for H in range(1, H_max + 1):
        errs_h = []
        for _ in range(n_trials):
            s_real, _ = env.reset()
            s_sim     = torch.FloatTensor(s_real).unsqueeze(0)

            # TODO: simula H pasos con el modelo, guardando s_H_simulado
            # TODO: ejecuta H pasos reales con las mismas acciones, guardando s_H_real
            # TODO: calcula la norma L2 entre s_H_real y s_H_simulado
            # Hint: guarda las acciones del loop real y reprodúcelas en el simulado

            errs_h.append(0.0)   # reemplaza con el error calculado
        errors.append(np.mean(errs_h))
    return errors

errors = measure_compounding_error(world_model, env, H_max=20)

# Visualizar
plt.figure(figsize=(8, 4))
plt.plot(range(1, 21), errors, 'o-', color='darkorchid', lw=2)
plt.xlabel('Horizonte H'); plt.ylabel('Error L2 promedio ||s_H||')
plt.title('Compounding Error vs Horizonte de Planificación')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('compounding_error.png', dpi=120); plt.show()


In [ ]:
# Agente Q-Network base (provisto) para optimizar con D_env ∪ D_model
class SimpleQAgent:
    def __init__(self, s_dim=STATE_DIM, n_act=N_ACTIONS, lr=1e-3, gamma=0.99):
        self.q_net = nn.Sequential(nn.Linear(s_dim, 128), nn.ReLU(),
                                   nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, n_act))
        self.opt = torch.optim.Adam(self.q_net.parameters(), lr=lr)
        self.gamma, self.n_act = gamma, n_act

    def select_action(self, s_tensor, eps=0.1):
        if np.random.rand() < eps:
            return torch.randint(0, self.n_act, (len(s_tensor),))
        with torch.no_grad():
            return self.q_net(s_tensor).argmax(dim=-1)

    def update(self, s, a, r, s2):
        q_val = self.q_net(s).gather(1, a.unsqueeze(1))
        with torch.no_grad():
            target = r + self.gamma * self.q_net(s2).max(dim=1, keepdim=True)[0]
        loss = nn.MSELoss()(q_val, target)
        self.opt.zero_grad(); loss.backward(); self.opt.step()

# ─── TODO 6: Branched Rollouts Cortos (D_env → D_model) ──────────────────────
def generate_mbpo_rollouts(wm, S_env, agent, H=3, n_starts=128):
    """
    Muestrea n_starts estados reales de S_env y simula H pasos con wm.
    Retorna tensores sintéticos: (S_mod, A_mod, R_mod, S2_mod)
    """
    idx = torch.randint(0, len(S_env), (n_starts,))
    s = S_env[idx].clone()
    S_m, A_m, R_m, S2_m = [], [], [], []

    for _ in range(H):
        # TODO: 1) selecciona acción a = agent.select_action(s, eps=0.2)
        # TODO: 2) convierte a en one-hot y predice (s_next, r_pred) con wm
        # TODO: 3) guarda (s, a, r_pred, s_next) en las listas y haz s = s_next
        pass

    return (torch.cat(S_m), torch.cat(A_m), torch.cat(R_m), torch.cat(S2_m))


In [ ]:
# ─── TODO 7: Entrenar Agente con MBPO (D_env ∪ D_model) ──────────────────────
def train_mbpo(wm, H_rollout=3, n_episodes=150, k_updates=10, batch_size=64):
    """Entrena SimpleQAgent combinando D_env y rollouts sintéticos D_model."""
    agent = SimpleQAgent()
    S_env, A_env, R_env, S2_env = collect_data(n_episodes=40)  # D_env inicial
    rewards_hist = []

    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep); ep_r = 0
        eps = max(0.05, 0.3 * (0.98 ** ep))
        for _ in range(500):
            s_t = torch.FloatTensor(s).unsqueeze(0)
            a = agent.select_action(s_t, eps=eps).item()
            s2, r, term, trunc, _ = env.step(a)
            r_tr = -10.0 if term else r
            S_env  = torch.cat([S_env, s_t])
            A_env  = torch.cat([A_env, torch.LongTensor([a])])
            R_env  = torch.cat([R_env, torch.FloatTensor([[r_tr]])])
            S2_env = torch.cat([S2_env, torch.FloatTensor(s2).unsqueeze(0)])
            s = s2; ep_r += r
            if term or trunc: break
        rewards_hist.append(ep_r)

        for _ in range(k_updates):
            # TODO: muestrea batch real de D_env. Si H_rollout > 0, genera
            # D_model con generate_mbpo_rollouts, concatena y llama agent.update
            pass
    return rewards_hist

# Comparar H = 0 (Model-Free), H = 3 (MBPO corto), H = 15 (Horizonte largo)
plt.figure(figsize=(8, 4))
for H in [0, 3, 15]:
    rews = train_mbpo(world_model, H_rollout=H, n_episodes=150)
    plt.plot(np.convolve(rews, np.ones(15)/15, mode='valid'), lw=2, label=f'MBPO H={H}')
plt.xlabel('Episodios reales'); plt.ylabel('Reward promedio (ventana=15)')
plt.title('MBPO: Efecto del Horizonte H (D_env ∪ D_model)'); plt.legend()
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('mbpo_horizontes.png', dpi=120); plt.show()
